In [1]:
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR  
import math 

base = "/kaggle/input/datasets/tait3r/asl-cleaned/islr_preprocessed"  
data = np.load(f"{base}/landmarks.npy")    
meta = pd.read_csv(f"{base}/meta.csv")
with open(f"{base}/selected_landmarks.json", "r") as f:
  groups = json.load(f)["groups"]

In [2]:
KEEP = ["lips", "eyes", "eyebrows", "left_hand", "right_hand"]
KEEP_IDX = np.concatenate([np.arange(*groups[g]) for g in KEEP])
HAND_IDX = np.concatenate([np.arange(*groups[g]) for g in ["left_hand", "right_hand"]])

N_LANDMARKS = len(KEEP_IDX)          # 134

In [3]:
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

MAX_LEN     = 64
CHANNELS    = len(KEEP_IDX) * 2          # 134 * 2 = 268
NUM_CLASSES = 250
BATCH_SIZE  = 128

def preprocess_clip(clip):
    x = np.asarray(clip, dtype=np.float32)[:, KEEP_IDX, :]      # (T, 134, 2)
    mean = np.nanmean(x, axis=(0, 1), keepdims=True)
    std  = np.nanstd(x) + 1e-6
    x = np.nan_to_num((x - mean) / std, nan=0.0)
    return x.reshape(len(x), CHANNELS)                           # (T, 268), unpadded

# drop clips longer than MAX_LEN
meta = meta[meta["length"] <= MAX_LEN].reset_index(drop=True)

# precompute, zero-padded to MAX_LEN
X = np.zeros((len(meta), MAX_LEN, CHANNELS), dtype=np.float16)
lengths = meta["length"].values
for i, (o, l) in enumerate(tqdm(zip(meta["offset"], lengths), total=len(meta))):
    X[i, :l] = preprocess_clip(data[o:o + l])
y = meta["label"].values

class SignDataset(Dataset):
    def __init__(self, idx):
        self.idx = idx
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        x = torch.from_numpy(X[j].astype(np.float32))            # (MAX_LEN, 268)
        pad_mask = torch.arange(MAX_LEN) >= lengths[j]           # True = padding
        return x, pad_mask, torch.tensor(y[j], dtype=torch.long)

tr_i, va_i = train_test_split(np.arange(len(meta)), test_size=0.2,
                              stratify=meta["label"], random_state=0)

train_loader = DataLoader(SignDataset(tr_i), batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, pin_memory=True,
                          persistent_workers=True)
val_loader   = DataLoader(SignDataset(va_i), batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=2, pin_memory=True, persistent_workers=True)

  0%|          | 0/80155 [00:00<?, ?it/s]

In [4]:
class CNNEmbedder(nn.Module):
    def __init__(self, in_ch=CHANNELS, d=192, k=5):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(in_ch, d, k, padding=k // 2),
            nn.Conv1d(d,     d, k, padding=k // 2),
            nn.Conv1d(d,     d, k, padding=k // 2),
        ])
        self.act  = nn.LeakyReLU(0.01)
        self.pool = nn.MaxPool1d(2, ceil_mode=True)

    def forward(self, x, pad_mask):
        # x: (B, T, C), pad_mask: (B, T) True = padding
        keep = (~pad_mask).unsqueeze(1).to(x.dtype)              # (B, 1, T)
        x = x.transpose(1, 2) * keep                              # (B, C, T)
        for conv in self.convs:
            x = self.act(conv(x)) * keep                          # re-zero padding after each conv

        # pool: padding can never win the max, and a pooled step is real if either frame was
        x = x.masked_fill(pad_mask.unsqueeze(1), float("-inf"))
        x = self.pool(x)
        pad_mask = self.pool(keep).squeeze(1) == 0                # (B, ceil(T/2))
        x = x.masked_fill(pad_mask.unsqueeze(1), 0.0)
        return x.transpose(1, 2), pad_mask                        # (B, T/2, d)


class CNNTransformer(nn.Module):
    def __init__(self, in_ch=CHANNELS, d=192, heads=4, n_cls=NUM_CLASSES, max_len=MAX_LEN):
        super().__init__()
        self.embedder = CNNEmbedder(in_ch, d)
        self.pos = nn.Parameter(torch.zeros(1, math.ceil(max_len / 2), d))  # 32 for max_len=64
        nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 2 * d, dropout=0.2, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(d, n_cls))

    def forward(self, x, pad_mask):
        h, pad_mask = self.embedder(x, pad_mask)
        h = h + self.pos[:, :h.size(1)]
        h = self.encoder(h, src_key_padding_mask=pad_mask)
        keep = (~pad_mask).unsqueeze(-1).to(h.dtype)
        h = (h * keep).sum(1) / keep.sum(1).clamp(min=1)          # masked mean
        return self.classifier(h)

In [7]:
import torch.nn.functional as F
import wandb

# On Kaggle: Add-ons -> Secrets -> add a secret named WANDB_API_KEY
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 0.05
device = "cuda" if torch.cuda.is_available() else "cpu"

wandb.init(
    entity="osherlock-carnegie-mellon-university",
    project="asl-signs",
    config={
        "model": "CNNTransformer",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "max_len": MAX_LEN,
        "channels": CHANNELS,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": 0.1,
    },
)

model = CNNTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader), pct_start=0.1
)
scaler = torch.amp.GradScaler(enabled=(device == "cuda"))

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    # --- train ---
    model.train()
    train_loss, train_correct, train_n = 0.0, 0, 0
    for x, pad_mask, yb in train_loader:
        x, pad_mask, yb = x.to(device, non_blocking=True), pad_mask.to(device), yb.to(device)

        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(x, pad_mask)
            loss = F.cross_entropy(logits, yb, label_smoothing=0.1)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss += loss.item() * len(yb)
        train_correct += (logits.argmax(1) == yb).sum().item()
        train_n += len(yb)

    # --- validate ---
    model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0
    with torch.no_grad(), torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
        for x, pad_mask, yb in val_loader:
            x, pad_mask, yb = x.to(device, non_blocking=True), pad_mask.to(device), yb.to(device)
            logits = model(x, pad_mask)
            val_loss += F.cross_entropy(logits, yb).item() * len(yb)
            val_correct += (logits.argmax(1) == yb).sum().item()
            val_n += len(yb)

    train_loss, train_acc = train_loss / train_n, train_correct / train_n
    val_loss, val_acc = val_loss / val_n, val_correct / val_n

    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "train/acc": train_acc,
        "val/loss": val_loss,
        "val/acc": val_acc,
        "lr": scheduler.get_last_lr()[0],
    })

    print(f"epoch {epoch:2d} | lr {scheduler.get_last_lr()[0]:.2e} | "
          f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        #torch.save(model.state_dict(), "/kaggle/working/best_model.pt")

wandb.run.summary["best_val_acc"] = best_acc
wandb.finish()
print(f"best val acc: {best_acc:.4f}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


epoch  1 | lr 1.81e-04 | train loss 5.0403 acc 0.0580 | val loss 4.0957 acc 0.1822
epoch  2 | lr 5.20e-04 | train loss 3.7579 acc 0.2508 | val loss 2.7462 acc 0.3657
epoch  3 | lr 8.60e-04 | train loss 3.0927 acc 0.3824 | val loss 2.4841 acc 0.4013
epoch  4 | lr 1.00e-03 | train loss 2.8045 acc 0.4551 | val loss 2.0640 acc 0.4901
epoch  5 | lr 9.98e-04 | train loss 2.5753 acc 0.5207 | val loss 1.7894 acc 0.5557
epoch  6 | lr 9.92e-04 | train loss 2.3995 acc 0.5708 | val loss 1.7202 acc 0.5753
epoch  7 | lr 9.83e-04 | train loss 2.2813 acc 0.6024 | val loss 1.5862 acc 0.6101
epoch  8 | lr 9.70e-04 | train loss 2.1829 acc 0.6285 | val loss 1.4829 acc 0.6326
epoch  9 | lr 9.53e-04 | train loss 2.1044 acc 0.6517 | val loss 1.4571 acc 0.6385
epoch 10 | lr 9.33e-04 | train loss 2.0268 acc 0.6735 | val loss 1.3925 acc 0.6516
epoch 11 | lr 9.10e-04 | train loss 1.9603 acc 0.6937 | val loss 1.3770 acc 0.6593
epoch 12 | lr 8.83e-04 | train loss 1.9027 acc 0.7100 | val loss 1.3826 acc 0.6579
epoc

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
lr,▂▅▇███████▇▇▇▇▇▆▆▆▅▅▅▄▄▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▂▃▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇███████████████
train/loss,█▆▅▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█████████████████████
val/loss,█▅▄▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.74524
epoch,40
lr,0.0
train/acc,0.98706
train/loss,1.07193


best val acc: 0.7452
